<a href="https://colab.research.google.com/github/Arobnett/HDX-sources-and-more-API-connection/blob/main/notebooks/03b_validate_geographic_keys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03b Validate Silver Geographic Keys

Runs strict ISO-3 validation and canonical country-name reconciliation after Silver cleaning and before Gold feature assembly. Invalid codes and country/code conflicts are retained in audit reports rather than entering the Gold join.

In [4]:
%cd /content/HDX-sources-and-more-API-connection  # Move to the cloned repository.

!git fetch origin  # Download the latest branch information from GitHub.
!git checkout geo-key-hardening  # Switch from main to the Silver cleanup branch.
!git pull origin geo-key-hardening  # Make sure the branch is fully current.

!ls src  # Confirm geographic_keys.py is now present.

[Errno 2] No such file or directory: '/content/HDX-sources-and-more-API-connection # Move to the cloned repository.'
/content
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
ls: cannot access 'src': No such file or directory


In [5]:
# Install the ISO country-reference library required by strict geographic validation.
!pip install -q pycountry

# Confirm the package is available.
import pycountry

print(f"pycountry loaded: {pycountry.__version__}")
print(f"USA lookup test: {pycountry.countries.get(alpha_3='USA').name}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 25.2 MB/s eta 0:00:00
pycountry loaded: 26.2.16
USA lookup test: United States


In [6]:
import sys  # Access Python's module search path.
from pathlib import Path  # Work with repository paths.

PROJECT_ROOT = Path("/content/HDX-sources-and-more-API-connection")  # Set repository root.

sys.path.insert(0, str(PROJECT_ROOT / "src"))  # Make shared modules importable.

from cleaning import clean_silver_directory  # Import the existing Silver cleaning pipeline.
from paths import CLEAN_DIR  # Import the cleaned Silver output directory.

summary_clean, rejected_clean = clean_silver_directory()  # Generate cleaned Silver CSVs.

display(summary_clean)  # Review the Silver cleaning results.

print(f"\nCleaned Silver CSVs created: {len(list(CLEAN_DIR.glob('*.csv')))}")

,source_file,output_name,status,input_rows,output_rows,rejected_rows,output_bytes,key_unique,duplicate_key_rows,missing_key_columns
0,civilian_targeting_events_and_fatalities__coun...,clean_civilian_targeting_country_month.csv,ok,1036352.0,6060.0,30336.0,163806.0,True,0.0,
1,demonstration_events__country_month_year.csv,clean_demonstration_events_country_month.csv,ok,1036352.0,6060.0,30336.0,145689.0,True,0.0,
2,political_violence_events_and_fatalities__coun...,clean_political_violence_country_month.csv,ok,1036352.0,6060.0,30336.0,168206.0,True,0.0,
3,who_covid_19_global_daily_data__country_month_...,clean_who_covid_country_month.csv,ok,573360.0,79.0,570971.0,8273.0,True,0.0,
4,gdacs_rss_information__country_month_year.csv,clean_gdacs_country_month.csv,ok,477.0,51.0,8.0,2267.0,True,0.0,
5,views_conflict_forecasts_country_month__countr...,clean_views_conflict_forecasts_country_month.csv,ok,1000.0,1000.0,0.0,46568.0,True,0.0,
6,inform_risk_index_trends__country_month_year.csv,clean_inform_risk_country_month.csv,ok,7640.0,22920.0,0.0,992941.0,True,0.0,
7,whs2026_datadownload_as_of_2026_08_02__country...,clean_whs2026_country_month.csv,ok,8299.0,16008.0,93.0,128865948.0,True,0.0,
8,worldriskindex_trend__country_month_year.csv,clean_worldriskindex_country_month.csv,ok,5018.0,60216.0,0.0,88025500.0,True,0.0,
9,hdx_colab_download_manifest__country_month_yea...,metadata_only_hdx_download_manifest.csv,metadata_skipped,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Cleaned Silver CSVs created: 9


In [7]:
import importlib  # Reload a module already imported in this runtime.
import geographic_keys  # Import the geographic validation module.

geographic_keys = importlib.reload(geographic_keys)  # Reload after installing pycountry.

from geographic_keys import validate_clean_directory  # Import the refreshed validator.
from paths import CLEAN_DIR, CLEAN_REPORTS_DIR  # Import canonical paths.

print("Geographic validator reloaded with pycountry.")

Geographic validator reloaded with pycountry.


In [8]:
summary_geo = validate_clean_directory(
    CLEAN_DIR,
    CLEAN_REPORTS_DIR
)  # Canonicalize country keys and write geographic audit reports.

display(summary_geo)  # Review accepted/rejected rows for every Silver source.

if summary_geo.empty:  # Stop if no cleaned files were found.
    raise ValueError("No cleaned Silver files were available for geographic validation.")

if not summary_geo["key_unique"].all():  # Block Gold if duplicate keys remain.
    raise ValueError(
        "Geographic-key validation failed uniqueness checks; inspect reports before Gold assembly."
    )

print("Silver geographic-key uniqueness validation passed.")

,source_file,input_rows,accepted_rows,rejected_rows,key_unique
0,clean_civilian_targeting_country_month.csv,6060,5356,704,True
1,clean_demonstration_events_country_month.csv,6060,5356,704,True
2,clean_gdacs_country_month.csv,51,38,13,True
3,clean_inform_risk_country_month.csv,22920,21360,1560,True
4,clean_political_violence_country_month.csv,6060,5356,704,True
5,clean_views_conflict_forecasts_country_month.csv,1000,853,147,True
6,clean_who_covid_country_month.csv,79,0,79,True
7,clean_whs2026_country_month.csv,16008,14616,1392,True
8,clean_worldriskindex_country_month.csv,60216,54912,5304,True


Silver geographic-key uniqueness validation passed.
